In [1]:
from pathlib import Path
from dataclasses import dataclass, asdict

@dataclass
class CFG:
    train_path: Path = Path("./data/train.csv")
    test_path: Path = Path("./data/test.csv")
    sub_path: Path = Path("./data/sample_submission.csv")

    num_fold: int = 5
    dev_mode: bool = False

    # Model parameters
    n_iter: int = 10000
    max_depth: int = -1
    num_leaves: int = 1024
    colsample_bytree: float = 0.7
    learning_rate: float = 0.02

    objective: str = 'l2'
    metric: str = 'rmse'
    verbosity: int = -1
    max_bin: int = 1024
    
    random_state: int = 42
    shuffle: bool = True
    encoded_columns_start: int = -91
    log_eval: int = 100
    early_stopping: int = 200
    
cfg = CFG() 
asdict(cfg)

{'train_path': PosixPath('data/train.csv'),
 'test_path': PosixPath('data/test.csv'),
 'sub_path': PosixPath('data/sample_submission.csv'),
 'num_fold': 5,
 'dev_mode': False,
 'n_iter': 10000,
 'max_depth': -1,
 'num_leaves': 1024,
 'colsample_bytree': 0.7,
 'learning_rate': 0.02,
 'objective': 'l2',
 'metric': 'rmse',
 'verbosity': -1,
 'max_bin': 1024,
 'random_state': 42,
 'shuffle': True,
 'encoded_columns_start': -91,
 'log_eval': 100,
 'early_stopping': 200}

In [ ]:
from IPython.display import display
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

re_dict = {}
re_dict['podc_dict'] = {
    'Mystery Matters': 0, 'Joke Junction': 1, 'Study Sessions': 2, 'Digital Digest': 3, 
    'Mind & Body': 4, 'Fitness First': 5, 'Criminal Minds': 6, 'News Roundup': 7, 
    'Daily Digest': 8, 'Music Matters': 9, 'Sports Central': 10, 'Melody Mix': 11, 
    'Game Day': 12, 'Gadget Geek': 13, 'Global News': 14, 'Tech Talks': 15, 
    'Sport Spot': 16, 'Funny Folks': 17, 'Sports Weekly': 18, 'Business Briefs': 19, 
    'Tech Trends': 20, 'Innovators': 21, 'Health Hour': 22, 'Comedy Corner': 23, 
    'Sound Waves': 24, 'Brain Boost': 25, "Athlete's Arena": 26, 'Wellness Wave': 27, 
    'Style Guide': 28, 'World Watch': 29, 'Humor Hub': 30, 'Money Matters': 31, 
    'Healthy Living': 32, 'Home & Living': 33, 'Educational Nuggets': 34, 
    'Market Masters': 35, 'Learning Lab': 36, 'Lifestyle Lounge': 37, 
    'Crime Chronicles': 38, 'Detective Diaries': 39, 'Life Lessons': 40, 
    'Current Affairs': 41, 'Finance Focus': 42, 'Laugh Line': 43, 
    'True Crime Stories': 44, 'Business Insights': 45, 'Fashion Forward': 46, 'Tune Time': 47
}
re_dict['genr_dict'] = {'True Crime': 0, 'Comedy': 1, 'Education': 2, 'Technology': 3, 'Health': 4, 'News': 5, 'Music': 6, 'Sports': 7, 'Business': 8, 'Lifestyle': 9}
re_dict['week_dict'] = {'Monday': 0, 'Tuesday': 1, 'Wednesday': 2, 'Thursday': 3, 'Friday': 4, 'Saturday': 5, 'Sunday': 6}
re_dict['time_dict'] = {'Morning': 10, 'Afternoon': 14, 'Evening': 17, 'Night': 21}
re_dict['sent_dict'] = {'Negative': 0, 'Neutral': 1, 'Positive': 2}


def preprocess_df(df):
    df['Episode_Num'] = df['Episode_Title'].str[8:].astype(int)  # Convert to int before log transform
    df = df.drop(columns=['Episode_Title'])

    # Convert categorical variables
    df['Genre'] = df['Genre'].replace(re_dict["genr_dict"])
    df['Podcast_Name'] = df['Podcast_Name'].replace(re_dict["podc_dict"])
    df['Publication_Day'] = df['Publication_Day'].replace(re_dict["week_dict"])
    df['Publication_Time'] = df['Publication_Time'].replace(re_dict["time_dict"])
    df['Episode_Sentiment'] = df['Episode_Sentiment'].replace(re_dict["sent_dict"])

    df['Host_Guest_Diff'] = df['Host_Popularity_percentage'] - df['Guest_Popularity_percentage']
    df['Host_Guest_Ratio'] = df['Host_Popularity_percentage'] / df['Guest_Popularity_percentage']

    if "Listening_Time_minutes" in df.columns:
        df['Listening_Episode_Diff'] = df['Episode_Length_minutes'] - df['Listening_Time_minutes']
        df['Listening_Episode_Ratio'] = df['Episode_Length_minutes'] / df['Listening_Time_minutes']
    
    # inf to NaN
    df = df.replace([float('inf'), -float('inf')], pd.NA)

    return df


df_train = pd.read_csv(cfg.train_path, index_col='id')
df_test = pd.read_csv(cfg.test_path, index_col='id')
df_sub = pd.read_csv(cfg.sub_path, index_col='id')

# is_dev_mode = False
# # is_dev_mode = True
# if is_dev_mode:
#     df_train = df_train.sample(10000, random_state=42)
#     df_test = df_test[:10]
#     df_sub = df_sub[:10]
    
df_train = preprocess_df(df_train)
df_test = preprocess_df(df_test)

# target_col = "Listening_Time_minutes"
# y_train = df_train[target_col].copy()
# df_train = df_train.drop(columns=[target_col])

display(df_train)
display(df_train.describe())

AttributeError: module 'pandas' has no attribute 'NAN'

In [10]:
df_train[df_train["Episode_Length_minutes"] > 120]

,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes,Episode_Num,Host_Guest_Diff,Host_Guest_Ratio,Listening_Episode_Diff,Listening_Episode_Ratio
id,,,,,,,,,,,,,,,
74270,20,120.64,3,49.78,5,10,50.54,1.0,1,119.73000,31,0.76,1.015267,0.91000,1.007600
87034,15,120.32,3,78.39,3,10,54.64,1.0,1,119.66000,31,-23.75,0.697028,0.66000,1.005516
101637,33,325.24,9,50.69,1,14,15.01,0.0,2,64.31981,16,-35.68,0.296114,260.92019,5.056607
168115,27,120.64,4,49.78,3,10,50.54,1.0,1,119.73000,78,0.76,1.015267,0.91000,1.007600
473895,26,120.06,7,78.98,5,17,56.64,1.0,1,119.67000,15,-22.34,0.717144,0.39000,1.003259
516498,46,120.37,9,79.40,3,14,54.66,0.0,1,119.44000,81,-24.74,0.688413,0.93000,1.007786
552181,26,120.73,7,78.42,4,17,54.24,1.0,1,119.67000,15,-24.18,0.691660,1.06000,1.008858
598106,38,120.93,0,48.73,4,10,72.96,2.0,2,70.90288,53,24.23,1.497230,50.02712,1.705572
688205,11,120.37,6,76.45,4,10,54.74,1.0,1,119.67000,70,-21.71,0.716024,0.70000,1.005849


In [17]:
# Please get only duplicate with Podcast_Name, Episode_Num
df_dup = df_train[df_train.duplicated(subset=['Podcast_Name', 'Host_Popularity_percentage'], keep=False)]
df_dup = df_dup.sort_values(['Podcast_Name', 'Host_Popularity_percentage', 'Guest_Popularity_percentage'])
df_dup

,Podcast_Name,Episode_Title,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes
id,,,,,,,,,,,
748552,Athlete's Arena,Episode 82,71.02,Sports,20.04,Friday,Afternoon,29.09,0.0,Neutral,56.09749
123261,Athlete's Arena,Episode 82,71.02,Music,20.04,Saturday,Afternoon,78.09,3.0,Positive,56.09749
707023,Athlete's Arena,Episode 82,71.02,Sports,20.04,Friday,Afternoon,78.09,0.0,Positive,56.09749
615974,Athlete's Arena,Episode 19,117.60,Sports,20.04,Sunday,Morning,NaN,2.0,Neutral,78.02501
88755,Athlete's Arena,Episode 63,72.54,Sports,20.06,Friday,Night,14.11,1.0,Negative,39.39000
...,...,...,...,...,...,...,...,...,...,...,...
162218,World Watch,Episode 71,44.50,News,99.89,Saturday,Evening,54.56,1.0,Negative,28.82105
381608,World Watch,Episode 57,16.63,News,99.89,Saturday,Evening,NaN,3.0,Negative,15.20660
491598,World Watch,Episode 50,74.06,News,99.89,Friday,Evening,NaN,2.0,Negative,50.76499


In [ ]:
# Please get only duplicate with Podcast_Name, Episode_Num
df_dup = df_train[df_train.duplicated(subset=['Podcast_Name', 'Episode_Num'], keep=False)]
df_dup = df_dup.sort_values(['Podcast_Name', 'Episode_Num', 'Host_Popularity_percentage', 'Guest_Popularity_percentage'])
df_dup

,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Episode_Num
id,,,,,,,,,,
2221,0,55.10,0,68.79,6,14,6.29,1.0,2,1
6639,0,85.75,0,96.60,0,21,9.86,0.0,0,1
12478,0,69.75,0,94.31,4,17,94.08,2.0,0,1
26451,0,63.84,0,95.01,6,10,22.62,0.0,1,1
33972,0,90.33,0,85.02,0,10,27.76,3.0,2,1
...,...,...,...,...,...,...,...,...,...,...
714536,47,50.09,6,67.47,2,10,53.58,0.0,0,100
715096,47,71.71,6,73.30,1,21,53.58,1.0,1,100
717445,47,7.20,6,38.86,0,17,30.73,1.0,1,100


Early stopping, best iteration is:
[9399]	training's rmse: 6.29499	valid_1's rmse: 12.5142
Validation score: 12.51421341907998